In [14]:
from typing import TypedDict, Annotated, List, Union
import operator
from langchain_core.messages import AIMessage
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.types import Command, interrupt
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import MemorySaver
import sqlite3
from langchain.tools import tool
from langchain_tavily import TavilySearch
import datetime
import re

In [15]:
# sqliite_saver=sqlite3.connect("SQLite_Database",check_same_thread=False)
# memory=SqliteSaver(sqliite_saver)
memory=MemorySaver()

In [16]:
class AgentState(TypedDict):
    input:str
    tool_result:Annotated[List, operator.add]
    errors:Union[Annotated[List,operator.add],None]
    chat_history:Annotated[List, operator.add]
    output:Union[str,None]

In [17]:
tools=[]

In [18]:
@tool
def get_system_time(format:str="%Y:%m:%d %H-%m-%S"):
    """Returns the system time in the specified format"""
    current_time = datetime.datetime.now()
    formatted_time=current_time.strftime(format)
    return formatted_time

search_tool=TavilySearch(search_depth="basic")

tools=[get_system_time,search_tool]

In [19]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

llm_with_tools=llm.bind_tools(tools=tools)

prompt_template=ChatPromptTemplate.from_messages([
    ("system","You are an AI Agent built using langgraph. You should not use the tools if you can answer the question using the information below\n"
    "Chat history: \n"
    "{chat_history}\n"
    "Tool results: This is the information you have collected by running the tools. Use this to answer the question insted of using tools.\n "
    "{tool_result}\n"
    "For the current process the following error was encountered by the llm, address this: \n"
    "{errors}"),
    ("human","{input}"),
    ("system","Use the Chat history, Tool results and error given above to answer the question. Do not use tools")
])

prompt_template_tool=ChatPromptTemplate.from_messages([
    ("system","You are an AI Agent built using langgraph, you are built to ensure that if the tools "
     " are not used if the user query can be used using the tool information and chat information given below\n"
    "Chat history: \n"
    "{chat_history}\n"
    "Tool results: This is the information you have collected by running the tools. Use this to answer the question insted of using tools.\n "
    "{tool_result}\n"),
    ("human","{input}"),
    ("system","Answer the human input using chat history and Tool results if you cannot answer stricly return the following string 'Toolcall required'")
])


In [20]:
responder_chain=prompt_template | llm_with_tools
validator_chain=prompt_template_tool| llm

def validator_node(state:AgentState):
    max_try=3
    current_state=state
    for i in range(max_try):
        try:
            result=validator_chain.invoke(current_state)
            if re.search(r"\Wrequired\W", result.content.lower()) and not re.search(r"\Wnot required\W", result.content.lower()):
                return Command(
                    goto="tool_executer",
                    update={"output":result}
                )
            else:
                return Command(
                    goto=END,
                    update={"output":result}
                )
        except Exception as e:
            last_error=e
            print(f"The following error occured during the invoke on the {i} attempt: {last_error}")
            error_msg=f"The following error occured during the last invoke :\n{last_error}"
            current_state["errors"]=[error_msg]

    result=f"The following error persists on the {max_try} invoke :\n{last_error}"
    return Command(
        goto=END,
        update={"errors": current_state.get("errors", None) + [result]}
    )


def responder_node(state:AgentState):
    max_try=3
    current_state=state
    for i in range(max_try):
        try:
            result=responder_chain.invoke(current_state)
            if isinstance(result,AIMessage) and hasattr(result,"tool_calls") and len(result.tool_calls)>0:
                if len(current_state["tool_result"])>0:
                    return Command(
                        goto="validator",
                        update={"output":result}
                    )
                else:
                    return Command(
                        goto="tool_executer",
                        update={"output":result}
                    )
            else:
                # print(f"*Secondtime*********{result.content}***********")
                return Command(
                    goto=END,
                    update={"output":result.content[0]["text"],"chat_history":[current_state["input"],result.content[0]["text"]]},
                    )
        except Exception as e:
            last_error=e
            print(f"The following error occured during the invoke on the {i} attempt: {last_error}")
            error_msg=f"The following error occured during the last invoke :\n{last_error}"
            current_state["errors"]=[error_msg]

    result=f"The following error persists on the {max_try} invoke :\n{last_error}"
    return Command(
        goto=END,
        update={"errors": current_state.get("errors", None) + [result]}
    )

def tool_node(state:AgentState):
    result=[]
    errors=[]
    last_AI_Message=state["output"]
    for tool in last_AI_Message.tool_calls:
        # print(last_AI_Message)
        # print(tool)
        # print(tool["name"])
        tool_name=tool["name"]
        tool_args=tool["args"]
        tool_to_be_executed=next((t for t in tools if t.name==tool_name),None)
        
        if tool_to_be_executed:
            tool_result=tool_to_be_executed.invoke(tool_args)
            # print(tool_result)
        else:
            errors.append(f"The tool that was called was {tool_name} which is not prestnt in the list of tools provided."
                        "Only use the the following tools:\n{tools}")

        result.append((tool_name,tool_result))
    
    if errors:
        # print("*********EXECUTED_ERRORS******")
        return Command(
            goto="responder",
            update={"tool_result":result, "errors":errors}
        )
    else:
        # print("*********EXECUTED_NON_ERRORS******")
        return Command(
            goto="responder",
            update={"tool_result":result}
        )

# def interrupt_node(state:AgentState):
#     human_response=interrupt("Do you want to call the tool? Press (Y/N):")
#     if human_response=="Y":
#         return Command(
#             goto="tool_executer"
#         )
#     else:
#         return Command(
#             goto=END
#         )

In [21]:
graph=StateGraph(AgentState)
graph.add_node("responder",responder_node)
graph.add_node("tool_executer",tool_node)
graph.add_node("validator",validator_node)

graph.set_entry_point('responder')

app=graph.compile(checkpointer=memory, interrupt_before=["tool_executer"])

config={"configurable":{"thread_id":3}}

In [22]:
query={
    "input":"How many days from today was the previous SpaceX rocket launch?",
    "tool_result":[],
    "errors":None,
    "chat_history":[],
    "output":None
}

# answer=app.invoke(Command(resume="Y"),query,config=config)

answer=app.stream(input=query,config=config,stream_mode="values")

In [23]:
for a in answer:
    print(a)

{'input': 'How many days from today was the previous SpaceX rocket launch?', 'tool_result': [], 'errors': None, 'chat_history': [], 'output': None}
{'input': 'How many days from today was the previous SpaceX rocket launch?', 'tool_result': [], 'errors': None, 'chat_history': [], 'output': AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "date of last SpaceX rocket launch"}'}, '__gemini_function_call_thought_signatures__': {'9ba6609d-deca-49ab-b858-bf8f553f34fd': 'CugIAQw51sd9BT6/qiYw7cXRGUO21AH1PGBz9T+4IadqDDhyhIrqFOzyZGh1Pd6JEf7Jcs8bKyTwdoqZmQn1BBRbzB55FRQsQiGaoBigQYvDLSllZ0tpnV/6XunFOApXsSPOfDl6nJYpOrjep7//6/jS+4Amd6d6tx5qc45C0Ov2LZ1ZnQ7l1u4wzS7XR8rPl2S6hmnFi5STnpMs1Y5QX/MshIa9+tAtcCyLpYuSukCNG3iDYIYHVdDHhTi0ivhkah2qLYa6fkl1804myRNdiyIB5Da0ZiE9vEEtF4L6NJx55AyuX3RGb4vT1avEE9jqI3DMadxtbNVCReSS4t5AqWJ8vxDOWDbNGhaeZ5O+iyW0qipw+dbnGtVeu5cvJ6T8srhxAkClMu0jiXA3V+XvnqgszXMLcSfYdqW6z+u/oLbFyBNf/keRTTVwNZuHFPpfx44aHikMOlV7Iuq9ApWhTSuQ5

In [24]:
print(app.get_state(config).next)

('tool_executer',)


In [25]:
answer=app.stream(None,config=config)

In [26]:
for a in answer:
    print(a)

{'tool_executer': {'tool_result': [('get_system_time', '2026-06-27'), ('tavily_search', {'query': 'date of last SpaceX rocket launch', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.spacex.com/launches', 'title': 'Launches - SpaceX', 'content': '*   [Starship](https://www.spacex.com/vehicles/starship). *   [Falcon 9](https://www.spacex.com/vehicles/falcon-9). *   [Falcon Heavy](https://www.spacex.com/vehicles/falcon-heavy). *   [Space Station](https://www.spacex.com/humanspaceflight/iss). *   [Earth Orbit](https://www.spacex.com/humanspaceflight/earth). *   [Mission](https://www.spacex.com/mission). [All Upcoming Launches](https://www.spacex.com/launches). [WATCH](https://www.spacex.com/launches/sl-17-40). | Starlink Mission | Falcon 9 | SLC-4E, California | Droneship | June 28, 2026 14:00 - 18:00 GMT+0 |. | Mission Clear Starship Flight Tests Human Spaceflight Space Station Resupply Science Commercial / International Rideshare Program Natio

In [27]:
print(app.get_state(config).next)

()
